# Myeloma, independent group: does lab sensitivity predict time to next treatment?

The QPOP myeloma paper has no patient outcomes. An independent group (Snijder lab, Zurich) published one:
Kropivsek et al., *Nature Cancer* 2023, "Ex vivo drug response heterogeneity reveals personalized therapeutic strategies for patients with multiple myeloma",
doi 10.1038/s43018-023-00544-9, CC BY. Their image-based assay ("pharmacoscopy") is not QPOP, but it tests the same premise.

Data, both CC BY and copied unchanged into `data/kropivsek2023_myeloma_tableS8_fig8b.csv`:
Supplementary Table 8 (therapy, time to next treatment or death, censoring) and Source Data Fig. 8b (iPCY = summed lab sensitivity to the drugs the patient then received).
Published claim (Fig. 8d): above-mean iPCY within each subcohort ("sensitive") → longer time to next treatment, log-rank P = 0.00763, HR 4.57 (1.77-11.8).

Survival statistics use the `lifelines` package.

In [1]:
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index
warnings.filterwarnings("ignore")

m = pd.read_csv("../data/kropivsek2023_myeloma_tableS8_fig8b.csv")
m["time"] = m.time_to_next_treatment_or_death_days
m["event"] = 1 - m.censored
m["immuno"] = m.immunotherapy_cohort.astype(bool)
m["transplant"] = m.therapy.str.contains("Mel", case=False)          # melphalan-containing regimen

def sensitive(d, how="mean"):
    return (d.ipcy > d.groupby("immuno").ipcy.transform(how)).astype(int)

def lr_p(d, s):
    return logrank_test(d.time[s == 1], d.time[s == 0], d.event[s == 1], d.event[s == 0]).p_value

m["sens"] = sensitive(m)
print(f"patients {len(m)}, events {int(m.event.sum())}, sensitive {m.sens.sum()}")
print(f"log-rank P (paper's split): {lr_p(m, m.sens):.5f}   (paper: 0.00763)")
for g, d in m.groupby("immuno"):
    print(f"  {'immunotherapy' if g else 'non-immunotherapy'} n={len(d)}: P = {lr_p(d, d.sens):.4f}")

patients 34, events 17, sensitive 13
log-rank P (paper's split): 0.00763   (paper: 0.00763)
  non-immunotherapy n=19: P = 0.0898
  immunotherapy n=15: P = 0.0222


In [2]:
fig = go.Figure()
for s, name in ((1, "lab-sensitive"), (0, "lab-resistant")):
    km = KaplanMeierFitter().fit(m.time[m.sens == s], m.event[m.sens == s])
    fig.add_trace(go.Scatter(x=km.survival_function_.index, y=km.survival_function_.iloc[:, 0], mode="lines",
                             line_shape="hv", name=f"{name} (n={int((m.sens == s).sum())})"))
fig.update_layout(xaxis_title="days", yaxis=dict(title="still on the same treatment", range=[0, 1.02]), height=420,
                  title="Reproduced Fig. 8d from the published data")
fig

## Robustness: cutoff, single patients, chance, and treatment intensity

In [3]:
checks = {}
checks["median instead of mean cutoff"] = lr_p(m, sensitive(m, "median"))
loo = []
for i in range(len(m)):
    d = m.drop(m.index[i])
    loo.append(lr_p(d, sensitive(d)))
checks["leave-one-out, largest P"] = max(loo)
import os
rng = np.random.default_rng(0); obs = lr_p(m, m.sens); hits = 0; B = int(os.environ.get("PPM_PERM_B", 5000))  # tests use fewer
for _ in range(B):
    d = m.copy(); d["ipcy"] = d.groupby("immuno").ipcy.transform(lambda x: rng.permutation(x.values))
    hits += lr_p(d, sensitive(d)) <= obs
checks["permutation P (iPCY shuffled within subcohort)"] = (hits + 1) / (B + 1)
nt = m[~m.transplant]
checks["no melphalan regimens, original labels (n=%d)" % len(nt)] = lr_p(nt, nt.sens)
checks["no melphalan regimens, cutoff recomputed"] = lr_p(nt, sensitive(nt))
for k, v in checks.items():
    print(f"{k:48s} {v:.4f}")
print("C-index of continuous iPCY (no cutoff):", round(concordance_index(m.time, m.ipcy, m.event), 2))
print("median days: melphalan regimens", m.time[m.transplant].median(), "| others", m.time[~m.transplant].median())

COX = {}
for strata in (None, ["transplant"], ["transplant", "immuno"]):
    cols = ["time", "event", "sens"] + (strata or [])
    c = CoxPHFitter().fit(m[cols].astype(float), "time", "event", strata=strata)
    r = c.summary.loc["sens"]
    COX[str(strata)] = (r["exp(coef)"], r["exp(coef) lower 95%"], r["exp(coef) upper 95%"], r["p"])
    print(f"Cox, strata {str(strata):24s} HR sensitive vs resistant {r['exp(coef)']:.2f} "
          f"({r['exp(coef) lower 95%']:.2f}-{r['exp(coef) upper 95%']:.2f}), P = {r['p']:.4f}")

median instead of mean cutoff                    0.0291
leave-one-out, largest P                         0.0423
permutation P (iPCY shuffled within subcohort)   0.0094
no melphalan regimens, original labels (n=25)    0.0033
no melphalan regimens, cutoff recomputed         0.0625
C-index of continuous iPCY (no cutoff): 0.64
median days: melphalan regimens 506.0 | others 210.0
Cox, strata None                     HR sensitive vs resistant 0.21 (0.06-0.74), P = 0.0150
Cox, strata ['transplant']           HR sensitive vs resistant 0.20 (0.06-0.74), P = 0.0158
Cox, strata ['transplant', 'immuno'] HR sensitive vs resistant 0.23 (0.05-1.02), P = 0.0536


## What this says

1. **The published result reproduces exactly** from the published data (log-rank P = 0.00763), and it is robust: a median cutoff, dropping any single patient, and a permutation test all keep it significant.
2. **Treatment intensity mostly does not explain it.** Melphalan (transplant) regimens last much longer (median 506 vs 210 days). Stratifying for them leaves the hazard ratio unchanged (about 0.2 for lab-sensitive vs lab-resistant). Dropping those 9 patients keeps the effect with the original labels (P = 0.003) but not if the cutoff is recomputed on the remaining 25 (P = 0.06). Stratifying for both transplant and immunotherapy subcohort makes it borderline (P about 0.05).
3. **It is small.** 34 patients and 17 events. The hazard ratio interval is wide, and the reported HR 4.57 depends on the estimation method: a Cox model gives about 4.8 with a wider interval, and in the immunotherapy subgroup a Cox model cannot be fitted at all because the groups separate completely.
4. **The continuous score is modest** (C-index about 0.64). The clear split comes from dichotomising at the subcohort mean.

Compared with QPOP: this independent, open study gives the most credible support found for the premise that a patient's cells in the lab can predict clinical response. It is one cohort, not yet replicated, and not a randomized test of guiding treatment by the assay.